In [ ]:
# ============================================================
# Thesis figures, publication ready (oesophageal cohort, V6) - 8 figures
# Run on MyDRE. Reads derived result CSVs + cohort_clean, writes PNGs.
# Viability figure removed; trajectory added value is now 4.7, ASA is 4.8.
# Requires: pandas, numpy, matplotlib, scikit-learn (seaborn optional)
# ============================================================
import os
for v in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','NUMEXPR_NUM_THREADS']:
    os.environ[v] = '1'
import numpy as np, pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid", context="paper")
    PALETTE = sns.color_palette("colorblind")
    HAS_SNS = True
except Exception:
    PALETTE = plt.rcParams['axes.prop_cycle'].by_key()['color']
    HAS_SNS = False

plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 300, "savefig.bbox": "tight",
    "savefig.facecolor": "white", "figure.facecolor": "white",
    "font.size": 11, "axes.titlesize": 12.5, "axes.titleweight": "bold",
    "axes.labelsize": 11, "xtick.labelsize": 10, "ytick.labelsize": 10,
    "legend.fontsize": 9.5, "axes.edgecolor": "#444444",
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Nimbus Roman", "DejaVu Serif", "serif"],
    "mathtext.fontset": "dejavuserif",
})

THESIS  = Path(r"Z:\Users\predicting recovery\AmanDeep\Thesis")
OUTPUTS = THESIS / "outputs"
DERIVED = THESIS / "data_derived"
FIG_DIR = OUTPUTS / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

def find_csv(name):
    hits = list(OUTPUTS.rglob(name)) + list(DERIVED.glob(name))
    if not hits:
        raise FileNotFoundError(f"{name} not found under {OUTPUTS} or {DERIVED}")
    return hits[0]

CAPTIONS = {
 "4.1": "Cross validated discrimination for the ASA baseline, the preoperative model, and the perioperative model, by outcome.",
 "4.2": "Adjusted odds ratios for major complications.",
 "4.3": "Adjusted odds ratios for prolonged stay.",
 "4.4": "Discrimination of the logistic model, the lasso, and the random forest on the perioperative features.",
 "4.5": "Decision curves for the perioperative models, showing net benefit against treating all or none.",
 "4.6": "Outcome rate by physiotherapy conclusion.",
 "4.7": "Change in discrimination when a trajectory feature is added to the surgical model.",
 "4.8": "Mean Charlson Comorbidity Index by ASA class.",
}

def save(fig, fignum, stem):
    path = FIG_DIR / f"{stem}.png"
    fig.savefig(path)
    plt.close(fig)
    print(f"  saved  Figure {fignum}  ->  {path.name}")
    print(f"         caption: {CAPTIONS[fignum]}")

OUT_LABELS = {
    "pulmonary": "Pulmonary", "cd_ge_IIIb": "Major complication",
    "reintervention": "Reintervention", "los_prolonged": "Prolonged stay",
    "textbook_failure": "Textbook outcome", "textbook_outcome": "Textbook outcome",
}
C_ASA, C_PRE, C_PERI = PALETTE[7], PALETTE[0], PALETTE[2]

print("seaborn:", HAS_SNS, "| figures ->", FIG_DIR)
print("\nfile name  ->  figure")
for n, stem in [("4.1","fig_4_1_discrimination"),("4.2","fig_4_2_forest_major"),
                ("4.3","fig_4_3_forest_prolonged"),("4.4","fig_4_4_model_family"),
                ("4.5","fig_4_5_decision_curves"),("4.6","fig_4_6_rate_by_conclusion"),
                ("4.7","fig_4_7_trajectory_added_value"),("4.8","fig_4_8_asa_vs_comorbidity")]:
    print(f"  {stem}.png  ->  Figure {n}")


In [ ]:
# Figure 4.1  Cross validated discrimination by information set.
s = pd.read_csv(find_csv("periop_vs_preop_summary.csv"))
order = ["cd_ge_IIIb","los_prolonged","pulmonary","reintervention","textbook_failure"]
if "outcome" in s.columns:
    s = s.set_index("outcome").reindex([o for o in order if o in set(s["outcome"])]).reset_index()
s["label"] = s["outcome"].map(OUT_LABELS).fillna(s["outcome"])
x = np.arange(len(s)); w = 0.26
fig, ax = plt.subplots(figsize=(7.4, 4.4))
ax.bar(x - w, s["ASA"],    w, label="ASA only",      color=C_ASA)
ax.bar(x,     s["preop"],  w, label="Preoperative",  color=C_PRE)
ax.bar(x + w, s["periop"], w, label="Perioperative", color=C_PERI)
ax.axhline(0.5, ls=":", color="grey", lw=1)
ax.text(len(s)-0.45, 0.505, "chance", color="grey", fontsize=8, ha="right")
ax.set_xticks(x); ax.set_xticklabels(s["label"], rotation=12, ha="right")
ax.set_ylabel("Cross validated AUC"); ax.set_ylim(0.45, 0.70)
ax.set_title("Discrimination by information set")
ax.legend(frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(0.5, 1.0))
for arr, off in [(s["ASA"], -w), (s["preop"], 0.0), (s["periop"], w)]:
    for xi, val in zip(x + off, arr): ax.text(xi, val+0.004, f"{val:.2f}", ha="center", fontsize=7.5)
if HAS_SNS: sns.despine(ax=ax)
save(fig, "4.1", "fig_4_1_discrimination")


In [ ]:
# Figures 4.2 and 4.3  Adjusted odds ratios, computed from cohort_clean.
# Factors whose 95% interval cannot be estimated (unstable, from collinearity or
# small numbers) are omitted from the plot and flagged in the printout.
from sklearn.linear_model import LogisticRegression

PREOP = ['age_at_surgery','sex_male','bmi','asascore','comlong','charlson_cci',
         'anamok_prior_surgery','immunosuppression','neoadj_received',
         'neoadj_chemoradiation','ct_stage','cn_stage','histology_adeno']
OPER_CORE = ['surgical_approach_mis','anastomosis_cervical','resection_transthoracic','conversion']
CONT  = ['age_at_surgery','bmi','charlson_cci','ct_stage','cn_stage']
FEATS = PREOP + OPER_CORE
LBL = {'age_at_surgery':'Age at surgery','sex_male':'Male sex','bmi':'Body mass index',
       'asascore':'ASA score','comlong':'Chronic pulmonary disease','charlson_cci':'Charlson index',
       'anamok_prior_surgery':'Prior abdominal surgery','immunosuppression':'Immunosuppression',
       'neoadj_received':'Neoadjuvant therapy','neoadj_chemoradiation':'Neoadjuvant chemoradiation',
       'ct_stage':'Clinical T stage','cn_stage':'Clinical N stage','histology_adeno':'Adenocarcinoma',
       'surgical_approach_mis':'Minimally invasive approach','anastomosis_cervical':'Cervical anastomosis',
       'resection_transthoracic':'Transthoracic resection','conversion':'Conversion to open'}

coh = pd.read_csv(find_csv("cohort_clean.csv"))

def adjusted_or(outcome):
    src = 'textbook_outcome' if outcome == 'textbook_failure' else outcome
    cc = coh[FEATS + [src]].dropna().copy()
    y = (1 - cc[src]).astype(int) if outcome == 'textbook_failure' else cc[src].astype(int)
    X = cc[FEATS].astype(float).copy()
    for c in CONT:
        if c in X.columns:
            mu, sd = X[c].mean(), X[c].std(ddof=0)
            X[c] = (X[c] - mu) / (sd if sd else 1.0)
    Xv = X.values
    lr = LogisticRegression(penalty='l2', C=1.0, solver='liblinear', max_iter=5000, random_state=42)
    lr.fit(Xv, y.values)
    beta = lr.coef_.ravel()
    Xd = np.column_stack([np.ones(len(Xv)), Xv])
    p = lr.predict_proba(Xv)[:, 1]
    Wd = p * (1 - p)
    cov = np.linalg.pinv(Xd.T @ (Xd * Wd[:, None]) + 1e-6 * np.eye(Xd.shape[1]))
    se = np.sqrt(np.clip(np.diag(cov)[1:], 0, None))
    rows = []
    for name, b, sderr in zip(FEATS, beta, se):
        orv = float(np.exp(b))
        if 0 < sderr < 5 and np.isfinite(sderr):
            lo, hi = float(np.exp(b - 1.96 * sderr)), float(np.exp(b + 1.96 * sderr))
        else:
            lo = hi = None
        rows.append((LBL[name], orv, lo, hi))
    return rows

XMIN, XMAX = 0.2, 8.0
forests = {("4.2","fig_4_2_forest_major"):     ("cd_ge_IIIb",    "Major complication (Clavien Dindo IIIb or higher)"),
           ("4.3","fig_4_3_forest_prolonged"): ("los_prolonged", "Prolonged length of stay (over 7 days)")}
for (fignum, stem), (oc, title) in forests.items():
    allrows = sorted(adjusted_or(oc), key=lambda r: r[1])
    print(f"\nFigure {fignum} odds ratios ({oc}):")
    for lab, orv, lo, hi in allrows[::-1]:
        ci = f"({lo:.2f}, {hi:.2f})" if lo is not None else "(interval not estimable, omitted from plot)"
        print(f"  {lab:<28} {orv:5.2f}  {ci}")
    rows = [r for r in allrows if r[2] is not None]
    dropped = [r[0] for r in allrows if r[2] is None]
    if dropped: print("  omitted from plot:", ", ".join(dropped))
    labels = [r[0] for r in rows]; yy = np.arange(len(rows))
    fig, ax = plt.subplots(figsize=(7.0, 5.8))
    ax.axvline(1.0, ls=":", color="grey", lw=1)
    for i, (lab, orv, lo, hi) in enumerate(rows):
        col = C_PERI if orv >= 1 else PALETTE[1]
        ax.plot([max(lo, XMIN), min(hi, XMAX)], [i, i], color=col, lw=1.6, zorder=1)
        ax.plot(min(max(orv, XMIN), XMAX), i, "o", color=col, ms=6, zorder=2)
    ax.set_yticks(yy); ax.set_yticklabels(labels)
    ax.set_xscale("log"); ax.set_xlim(XMIN, XMAX)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:g}"))
    ax.set_xticks([0.25, 0.5, 1, 2, 4, 8])
    ax.set_xlabel("Adjusted odds ratio (95% CI, log scale)")
    ax.set_title(title)
    if HAS_SNS: sns.despine(ax=ax, left=True)
    save(fig, fignum, stem)


In [ ]:
# Figure 4.4  Model family comparison on the perioperative features.
try:
    mf = pd.read_csv(find_csv("model_family_panel.csv")); src = "csv"
except FileNotFoundError:
    from sklearn.linear_model import LogisticRegression
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.preprocessing import StandardScaler
    from sklearn.compose import ColumnTransformer
    from sklearn.pipeline import Pipeline
    from sklearn.model_selection import cross_val_predict, StratifiedKFold
    from sklearn.metrics import roc_auc_score
    def pipe(est):
        pre = ColumnTransformer([('s', StandardScaler(), [c for c in CONT if c in FEATS])], remainder='passthrough')
        return Pipeline([('t', pre), ('m', est)])
    ests = {"logistic (tuned)": LogisticRegression(penalty='l2', C=1.0, solver='liblinear', max_iter=5000, random_state=42),
            "lasso": LogisticRegression(penalty='l1', C=1.0, solver='liblinear', max_iter=5000, random_state=42),
            "random forest": RandomForestClassifier(n_estimators=200, n_jobs=1, random_state=42)}
    recs = []
    for oc in ["pulmonary","cd_ge_IIIb","reintervention","los_prolonged","textbook_failure"]:
        s2 = 'textbook_outcome' if oc == 'textbook_failure' else oc
        cc = coh[FEATS + [s2]].dropna()
        y = (1 - cc[s2]).astype(int) if oc == 'textbook_failure' else cc[s2].astype(int)
        if y.nunique() < 2: continue
        row = {"outcome": oc}
        for nm, est in ests.items():
            pr = cross_val_predict(pipe(est), cc[FEATS], y, cv=StratifiedKFold(5, shuffle=True, random_state=42), method='predict_proba')[:, 1]
            row[nm] = round(roc_auc_score(y, pr), 3)
        recs.append(row)
    mf = pd.DataFrame(recs); src = "recomputed"
oc0 = mf.columns[0]
order = ["cd_ge_IIIb","los_prolonged","pulmonary","reintervention","textbook_failure"]
mf = mf.set_index(oc0).reindex([o for o in order if o in set(mf[oc0])]).reset_index()
mf["label"] = mf[oc0].map(OUT_LABELS).fillna(mf[oc0])
models = [c for c in mf.columns if c not in (oc0, "label")]
x = np.arange(len(mf)); w = 0.8 / len(models)
fig, ax = plt.subplots(figsize=(8.2, 4.4))
pal4 = [PALETTE[0], PALETTE[7], PALETTE[2], PALETTE[3]]
for i, m in enumerate(models):
    ax.bar(x + i*w, mf[m], w, label=m.replace("_", " ").title(), color=pal4[i % 4])
ax.axhline(0.5, ls=":", color="grey", lw=1)
ax.set_xticks(x + w*(len(models)-1)/2); ax.set_xticklabels(mf["label"], rotation=12, ha="right")
ax.set_ylabel("Cross validated AUC"); ax.set_ylim(0.45, 0.70)
ax.set_title("Model family comparison (perioperative features)")
ax.legend(frameon=False, ncol=len(models), loc="upper center")
if HAS_SNS: sns.despine(ax=ax)
print("source:", src)
save(fig, "4.4", "fig_4_4_model_family")


In [ ]:
# Figure 4.5  Decision curves for the perioperative models.
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_predict, StratifiedKFold

def periop_pipe():
    pre = ColumnTransformer([('s', StandardScaler(), [c for c in CONT if c in FEATS])], remainder='passthrough')
    lr = LogisticRegression(penalty='l2', C=1.0, solver='liblinear', max_iter=5000, random_state=42)
    return Pipeline([('t', pre), ('lr', lr)])

def cv_probs(X, y):
    return cross_val_predict(periop_pipe(), X, y, cv=StratifiedKFold(5, shuffle=True, random_state=42), method='predict_proba')[:, 1]

def net_benefit(y, p, ts):
    y = np.asarray(y); p = np.asarray(p); n = len(y); prev = y.mean()
    nb_m = [np.sum((p >= t) & (y == 1))/n - np.sum((p >= t) & (y == 0))/n*(t/(1-t)) for t in ts]
    nb_all = [prev - (1-prev)*(t/(1-t)) for t in ts]
    return np.array(nb_m), np.array(nb_all)

dca_outcomes = ['pulmonary','cd_ge_IIIb','reintervention','los_prolonged','textbook_failure']
ts = np.arange(0.05, 0.51, 0.05)
fig, axes = plt.subplots(2, 3, figsize=(11, 6.4)); axes = axes.ravel()
for ax, oc in zip(axes, dca_outcomes):
    src = 'textbook_outcome' if oc == 'textbook_failure' else oc
    cc = coh[FEATS + [src]].dropna()
    y = (1 - cc[src]).astype(int) if oc == 'textbook_failure' else cc[src].astype(int)
    if y.nunique() < 2:
        ax.set_visible(False); continue
    p = cv_probs(cc[FEATS], y.values)
    nb_m, nb_all = net_benefit(y.values, p, ts)
    ax.plot(ts, nb_m, marker='o', ms=4, color=C_PERI, label='Perioperative model')
    ax.plot(ts, nb_all, ls='--', color='grey', label='Treat all')
    ax.axhline(0, ls=':', color='black', lw=1, label='Treat none')
    ax.set_title(OUT_LABELS.get(oc, oc)); ax.set_xlabel("Threshold probability")
    ax.set_ylabel("Net benefit")
    ax.set_ylim(min(-0.02, float(nb_m.min())*1.1), max(0.06, float(nb_m.max())*1.2))
    if HAS_SNS: sns.despine(ax=ax)
axes[-1].axis('off')
axes[0].legend(frameon=False, loc='upper right')
fig.suptitle("Decision curve analysis, perioperative models", y=1.0, fontsize=12.5, fontweight='bold')
fig.tight_layout()
save(fig, "4.5", "fig_4_5_decision_curves")


In [ ]:
# Figure 4.6  Outcome rate by physiotherapy conclusion.
try:
    rq1 = pd.read_csv(find_csv("rq1_conclusion.csv"))
except FileNotFoundError:
    DEFAULT = {"pulmonary": (0.220, 0.280), "cd_ge_IIIb": (0.121, 0.176), "los_prolonged": (0.528, 0.710)}
    rq1 = pd.DataFrame([{"outcome": k, "rate_low": lo, "rate_high": hi} for k, (lo, hi) in DEFAULT.items()])
order = ["pulmonary","cd_ge_IIIb","los_prolonged"]
rq1 = rq1[rq1["outcome"].isin(order)].set_index("outcome").reindex(order).reset_index()
rq1["label"] = rq1["outcome"].map(OUT_LABELS).fillna(rq1["outcome"])
x = np.arange(len(rq1)); w = 0.36
fig, ax = plt.subplots(figsize=(7.0, 4.2))
ax.bar(x - w/2, rq1["rate_low"]*100,  w, label="Low conclusion",  color=PALETTE[0])
ax.bar(x + w/2, rq1["rate_high"]*100, w, label="High conclusion", color=PALETTE[3])
ax.set_xticks(x); ax.set_xticklabels(rq1["label"])
ax.set_ylabel("Observed outcome rate (%)")
ax.set_title("Outcome rate by physiotherapy risk conclusion")
ax.legend(frameon=False)
for xi, val in zip(x - w/2, rq1["rate_low"]*100):  ax.text(xi, val+0.6, f"{val:.1f}", ha="center", fontsize=8)
for xi, val in zip(x + w/2, rq1["rate_high"]*100): ax.text(xi, val+0.6, f"{val:.1f}", ha="center", fontsize=8)
if HAS_SNS: sns.despine(ax=ax)
save(fig, "4.6", "fig_4_6_rate_by_conclusion")


In [ ]:
# Figure 4.7  Change in discrimination when a trajectory feature is added to the surgical model.
ICF_NAME = {'ENR':'Energy','ATT':'Concentration','STM':'Mood','ADM':'Respiration',
            'INS':'Exercise tolerance','MBW':'Weight maintenance','FAC':'Walking',
            'ETN':'Eating','BER':'Work and study'}
def pretty_feat(f):
    parts = str(f).split('_')
    name = ICF_NAME.get(parts[0], parts[0])
    rest = ' '.join(parts[1:])
    return (name + ' ' + rest).strip()

tav = pd.read_csv(find_csv("trajectory_added_value.csv"))
gain_col = "gain" if "gain" in tav.columns else tav.columns[-1]
outs = ["pulmonary", "cd_ge_IIIb", "los_prolonged"]
fig, axes = plt.subplots(1, 3, figsize=(11, 4.8), sharex=True)
for ax, oc in zip(axes, outs):
    d = tav[tav["outcome"] == oc].sort_values(gain_col)
    feats = d["feature"].map(pretty_feat)
    cols = [PALETTE[2] if g >= 0 else PALETTE[1] for g in d[gain_col]]
    ax.barh(feats, d[gain_col], color=cols)
    ax.axvline(0, color="black", lw=1)
    ax.set_title(OUT_LABELS.get(oc, oc)); ax.set_xlabel("Change in AUC")
    if HAS_SNS: sns.despine(ax=ax)
fig.suptitle("Added value of preoperative trajectory features over the surgical model", y=1.02, fontsize=12.5, fontweight="bold")
fig.tight_layout()
save(fig, "4.7", "fig_4_7_trajectory_added_value")


In [ ]:
# Figure 4.8  Mean Charlson Comorbidity Index by ASA class (classes 1 to 3).
try:
    av = pd.read_csv(find_csv("asa_vs_comorbidity.csv"))
    if "asascore" not in av.columns:
        av = av.rename(columns={av.columns[0]: "asascore"})
except FileNotFoundError:
    cc = coh.dropna(subset=["asascore"]).copy()
    cc["asascore"] = cc["asascore"].round().astype(int)
    av = cc.groupby("asascore").agg(mean_charlson=("charlson_cci", "mean"),
                                    pct_COPD=("comlong", lambda z: 100*z.mean())).reset_index()
av = av[av["asascore"].isin([1, 2, 3])]
fig, ax = plt.subplots(figsize=(7.0, 4.2))
bars = ax.bar(av["asascore"].astype(int).astype(str), av["mean_charlson"], color=C_PRE, width=0.55)
ax.set_xlabel("ASA physical status class"); ax.set_ylabel("Mean Charlson Comorbidity Index")
ax.set_title("Comorbidity burden rises with ASA class")
for b, val in zip(bars, av["mean_charlson"]): ax.text(b.get_x()+b.get_width()/2, val+0.02, f"{val:.2f}", ha="center", fontsize=9)
if "pct_COPD" in av.columns:
    ax2 = ax.twinx()
    ax2.plot(av["asascore"].astype(int).astype(str), av["pct_COPD"], marker="o", color=PALETTE[3], lw=2, label="COPD prevalence")
    ax2.set_ylabel("COPD prevalence (%)", color=PALETTE[3]); ax2.tick_params(axis="y", labelcolor=PALETTE[3])
    ax2.grid(False); ax2.set_ylim(0, max(av["pct_COPD"])*1.3)
if HAS_SNS: sns.despine(ax=ax, right=False)
save(fig, "4.8", "fig_4_8_asa_vs_comorbidity")
print("\nAll figures written to", FIG_DIR)


In [ ]:
# Figure 4.9  Calibration of the perioperative models (observed vs predicted risk).
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.calibration import calibration_curve

PREOP = ['age_at_surgery','sex_male','bmi','asascore','comlong','charlson_cci',
         'anamok_prior_surgery','immunosuppression','neoadj_received',
         'neoadj_chemoradiation','ct_stage','cn_stage','histology_adeno']
OPER_CORE = ['surgical_approach_mis','anastomosis_cervical','resection_transthoracic','conversion']
CONT  = ['age_at_surgery','bmi','charlson_cci','ct_stage','cn_stage']
FEATS = PREOP + OPER_CORE
coh = pd.read_csv(find_csv("cohort_clean.csv"))

CAPTIONS["4.9"] = ("Calibration of the perioperative models, showing observed outcome "
                   "frequency against predicted risk across quantile bins, with the "
                   "calibration slope per outcome. The dotted line marks perfect calibration.")

def periop_pipe():
    pre = ColumnTransformer([('s', StandardScaler(), [c for c in CONT if c in FEATS])], remainder='passthrough')
    lr  = LogisticRegression(penalty='l2', C=1.0, solver='liblinear', max_iter=5000, random_state=42)
    return Pipeline([('t', pre), ('lr', lr)])

def cv_probs(X, y):
    return cross_val_predict(periop_pipe(), X, y,
                             cv=StratifiedKFold(5, shuffle=True, random_state=42),
                             method='predict_proba')[:, 1]

def calibration_slope(y, p):
    eps = 1e-6; p = np.clip(p, eps, 1 - eps)
    logit = np.log(p / (1 - p)).reshape(-1, 1)
    return float(LogisticRegression(C=1e8, solver='lbfgs', max_iter=3000).fit(logit, y).coef_[0, 0])

cal_outcomes = ['pulmonary','cd_ge_IIIb','reintervention','los_prolonged','textbook_failure']
fig, axes = plt.subplots(2, 3, figsize=(11, 6.4)); axes = axes.ravel()
for ax, oc in zip(axes, cal_outcomes):
    src = 'textbook_outcome' if oc == 'textbook_failure' else oc
    cc = coh[FEATS + [src]].dropna()
    y = (1 - cc[src]).astype(int) if oc == 'textbook_failure' else cc[src].astype(int)
    if y.nunique() < 2:
        ax.set_visible(False); continue
    p = cv_probs(cc[FEATS], y.values)
    frac_pos, mean_pred = calibration_curve(y.values, p, n_bins=8, strategy='quantile')
    slope = calibration_slope(y.values, p)
    ax.plot([0, 1], [0, 1], ls=':', color='grey', lw=1, label='Ideal')
    ax.plot(mean_pred, frac_pos, marker='o', ms=4, color=C_PERI, label='Perioperative model')
    ax.set_title(f"{OUT_LABELS.get(oc, oc)}  (slope {slope:.2f})")
    ax.set_xlabel("Predicted probability"); ax.set_ylabel("Observed frequency")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect('equal', 'box')
    if HAS_SNS: sns.despine(ax=ax)
axes[-1].axis('off')
axes[0].legend(frameon=False, loc='upper left')
fig.suptitle("Calibration of the perioperative models", y=1.02, fontsize=12.5, fontweight="bold")
fig.tight_layout()
save(fig, "4.9", "fig_4_9_calibration")